# NB8

Robustness and reviewer-response pack: cutoff, HVG, no-priors, purity, mito, and Visium checks.

In [ ]:
# Robustness and sensitivity pack

import os, re, gc, json, time, warnings, traceback, glob
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import scipy.stats as stats
from sklearn.decomposition import NMF
from scipy.optimize import linear_sum_assignment

import matplotlib
matplotlib.rcParams.update({
    "font.family": "Arial", "font.size": 8,
    "axes.titlesize": 9, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "figure.dpi": 150,
    "savefig.dpi": 1200, "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05, "axes.linewidth": 0.8,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_style("ticks")
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

try:
    import statsmodels.api as sm
    HAS_SM = True
except ImportError:
    HAS_SM = False

warnings.filterwarnings("ignore")
np.random.seed(42)
sc.settings.verbosity = 1

# 0) Paths + global config
BASE_DIR = Path(os.environ.get("MES_BASE_DIR", "."))
RAW_DIR  = BASE_DIR / "Raw Data"
PROC_DIR = BASE_DIR / "Process Data"
MANUSCRIPT_DIR = (BASE_DIR / "Manuscript data") if (BASE_DIR / "Manuscript data").exists() else (BASE_DIR / "Manuscript Data")
FIG_DIR = MANUSCRIPT_DIR / "Figures" / "Revision"
TAB_DIR = MANUSCRIPT_DIR / "Tables" / "Revision"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

AIM1_DIR = PROC_DIR / "aim1_thymus"
NEG_DIR  = PROC_DIR / "negctrl"
AIM2_DIR = PROC_DIR / "aim2_microglia"

def _first_existing(candidates, label):
    for p in candidates:
        if p.exists():
            return p
    return None

def _glob_first(root, patterns, label):
    if not root.exists():
        return None
    for pat in patterns:
        hits = sorted(root.rglob(pat))
        if hits:
            return hits[0]
    return None

# Try canonical NB2 outputs first; fall back to glob if missing.
TS_QC = _first_existing([AIM1_DIR / "TS_Thymus_filtered__qc.h5ad"], "TS_QC") or \
        _glob_first(PROC_DIR, ["*TS*Thymus*qc.h5ad", "TS_Thymus*.h5ad"], "TS_QC")
LAV_QC = _first_existing([AIM1_DIR / "GSE144870_Lavaert__qc.h5ad"], "LAV_QC") or \
         _glob_first(PROC_DIR, ["*Lavaert*qc.h5ad", "*GSE144870*.h5ad"], "LAV_QC")
LE_QC = _first_existing([AIM1_DIR / "GSE139042_Le__qc.h5ad"], "LE_QC") or \
        _glob_first(PROC_DIR, ["*Le*qc.h5ad", "*GSE139042*.h5ad"], "LE_QC")
BLOOD_QC = _first_existing([NEG_DIR / "TS_Blood_NegCtrl_Myeloid__qc.h5ad"], "BLOOD") or \
           _glob_first(PROC_DIR, ["*Blood*Myeloid*qc.h5ad", "*Blood_NegCtrl*.h5ad"], "BLOOD")
MAIN_T1  = None   # resolved by _find_gene_weights() below

N_BOOT = 2000
N_PERM = 5000
SEED   = 42

# Run-control flags. Set to False to skip slow analyses.
RUN_SLOW_NMF      = True   # A8 (HVG sweep) and A9 (no-priors): ~20-40 min
RUN_VISIUM        = True   # A12: depends on disk speed, ~5-15 min

master_log = []
def log(msg, tag=None):
    s = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(s, flush=True)
    master_log.append({"time": time.strftime("%H:%M:%S"), "tag": tag or "", "msg": msg})

def save_fig(fig, name, kind="Supp_Rev"):
    out = FIG_DIR / f"{kind}_{name}.png"
    fig.savefig(out, dpi=1200, bbox_inches="tight")
    plt.close(fig)
    log(f"FIG saved: {out.name}", tag="fig")

def save_xlsx(sheets, name):
    if isinstance(sheets, pd.DataFrame):
        sheets = {"Sheet1": sheets}
    out = TAB_DIR / f"Supp_Rev_{name}.xlsx"
    with pd.ExcelWriter(out, engine="openpyxl") as w:
        wrote = False
        for sn, df in sheets.items():
            if df is None or len(df) == 0:
                pd.DataFrame({"note": ["no data"]}).to_excel(w, index=False, sheet_name=str(sn)[:31])
            else:
                df.to_excel(w, index=False, sheet_name=str(sn)[:31])
                wrote = True
    log(f"TAB saved: {out.name}{'' if wrote else ' (empty)'}", tag="tab")

def bh_fdr(pvals):
    p = np.asarray(pvals, float)
    out = np.full_like(p, np.nan)
    m = np.isfinite(p)
    if m.sum() == 0: return out
    pv = p[m]; n = pv.size
    order = np.argsort(pv)
    ranked = pv[order]
    q = ranked * n / (np.arange(1, n+1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out_m = np.empty_like(pv)
    out_m[order] = np.clip(q, 0, 1)
    out[m] = out_m
    return out

def sig_stars(p):
    if pd.isna(p) or not np.isfinite(p): return "ns"
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def safe_corr_with_ci(x, y, method="spearman", n_boot=N_BOOT, seed=SEED, min_n=5):
    """Spearman or Pearson with bootstrap 95% CI."""
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy()
    y = pd.to_numeric(pd.Series(y), errors="coerce").to_numpy()
    m = np.isfinite(x) & np.isfinite(y)
    out = {"r": np.nan, "p": np.nan, "n": int(m.sum()), "ci_lo": np.nan, "ci_hi": np.nan}
    if m.sum() < min_n: return out
    xa, ya = x[m], y[m]; n = xa.size
    if method == "spearman":
        r, p = stats.spearmanr(xa, ya)
    else:
        r, p = stats.pearsonr(xa, ya)
    out["r"] = float(r); out["p"] = float(p)
    rng = np.random.RandomState(seed)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.randint(0, n, n)
        if method == "spearman":
            boots[i] = stats.spearmanr(xa[idx], ya[idx])[0]
        else:
            boots[i] = stats.pearsonr(xa[idx], ya[idx])[0]
    fb = boots[np.isfinite(boots)]
    if len(fb) > 10:
        out["ci_lo"] = float(np.percentile(fb, 2.5))
        out["ci_hi"] = float(np.percentile(fb, 97.5))
    return out

def weighted_pearson(x, y, w):
    """Weighted Pearson with Fisher-z approximate p-value using effective N."""
    x = np.asarray(x, float); y = np.asarray(y, float); w = np.asarray(w, float)
    m = np.isfinite(x) & np.isfinite(y) & np.isfinite(w) & (w > 0)
    if m.sum() < 5: return np.nan, np.nan, 0
    xa, ya, wa = x[m], y[m], w[m]
    mx = np.average(xa, weights=wa); my = np.average(ya, weights=wa)
    cov = np.average((xa - mx) * (ya - my), weights=wa)
    vx = np.average((xa - mx) ** 2, weights=wa)
    vy = np.average((ya - my) ** 2, weights=wa)
    if vx <= 0 or vy <= 0: return np.nan, np.nan, int(m.sum())
    r = cov / np.sqrt(vx * vy)
    n_eff = (wa.sum() ** 2) / np.sum(wa ** 2)
    if n_eff <= 3: return float(r), np.nan, int(n_eff)
    z = np.arctanh(np.clip(r, -0.999999, 0.999999))
    se = 1.0 / np.sqrt(n_eff - 3)
    p = 2.0 * (1.0 - stats.norm.cdf(abs(z) / se))
    return float(r), float(p), int(n_eff)

def random_effects_meta(r_list, n_list):
    """DerSimonian-Laird random-effects on Fisher-z transformed correlations."""
    items = [(r, n) for r, n in zip(r_list, n_list)
             if pd.notna(r) and pd.notna(n) and float(n) > 3]
    if not items:
        return {"pooled_r": np.nan, "ci_lo": np.nan, "ci_hi": np.nan,
                "I2": np.nan, "tau2": np.nan, "Q": np.nan, "Q_p": np.nan, "k": 0}
    r = np.array([x[0] for x in items], float)
    n = np.array([x[1] for x in items], float)
    z = np.arctanh(np.clip(r, -0.999999, 0.999999))
    v = 1.0 / (n - 3.0)
    w = 1.0 / v
    z_fixed = np.sum(w * z) / np.sum(w)
    Q = float(np.sum(w * (z - z_fixed) ** 2))
    df = max(0, len(z) - 1)
    C = np.sum(w) - (np.sum(w**2) / np.sum(w))
    tau2 = max(0.0, (Q - df) / max(C, 1e-12))
    w_re = 1.0 / (v + tau2)
    z_re = float(np.sum(w_re * z) / np.sum(w_re))
    se = float(1.0 / np.sqrt(np.sum(w_re)))
    I2 = 0.0 if Q <= df else float(max(0.0, (Q - df) / max(Q, 1e-12)) * 100.0)
    Q_p = float(1.0 - stats.chi2.cdf(Q, df)) if df > 0 else np.nan
    return {
        "pooled_r": float(np.tanh(z_re)),
        "ci_lo": float(np.tanh(z_re - 1.96 * se)),
        "ci_hi": float(np.tanh(z_re + 1.96 * se)),
        "I2": I2, "tau2": float(tau2), "Q": Q, "Q_p": Q_p, "k": int(len(z))
    }

def _upper_map(var_names):
    return {str(v).upper(): str(v) for v in var_names}

def genes_present(adata, genes):
    m = _upper_map(adata.var_names)
    out = []
    for g in genes:
        gg = str(g).upper()
        if gg in m and m[gg] not in out:
            out.append(m[gg])
    return out

def score_geneset(adata, genes, name, min_genes=3):
    present = genes_present(adata, genes)
    if len(present) < min_genes:
        adata.obs[name] = np.nan
        return 0
    try:
        sc.tl.score_genes(adata, present, score_name=name, use_raw=False)
        return len(present)
    except Exception as e:
        adata.obs[name] = np.nan
        return 0

def looks_log1p(adata, n=2000):
    if adata.n_obs == 0 or adata.n_vars == 0: return False
    X = adata.X
    if sp.issparse(X):
        v = X.data
        if v.size == 0: return False
        v = v[:min(v.size, n)]
    else:
        v = np.asarray(X).ravel()[:n]
    frac_nonint = np.mean(np.abs(v - np.round(v)) > 1e-6)
    return (np.nanmax(v) < 25) and (frac_nonint > 0.2)

def ensure_counts(a):
    if "counts" not in a.layers:
        a.layers["counts"] = a.X.copy()
    if not sp.issparse(a.layers["counts"]):
        a.layers["counts"] = sp.csr_matrix(a.layers["counts"])

def prep_for_scoring(adata):
    if adata.n_obs == 0 or adata.n_vars == 0: return
    ensure_counts(adata)
    if looks_log1p(adata): return
    adata.X = adata.layers["counts"].copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

def detect_donor_col(obs):
    cols = list(map(str, obs.columns))
    for cand in ["donor_id","Donor ID","donor","DonorID","patient_id","PatientID",
                 "subject_id","individual","Donor","Subject","participant_id","donor_id_clean"]:
        if cand in cols: return cand
    low = {c.lower(): c for c in cols}
    for key in ["donor", "subject", "patient", "individual", "participant", "case"]:
        for lc, orig in low.items():
            if key in lc and ("id" in lc or lc.endswith(key)): return orig
    return None

def detect_pmi_col(obs):
    for cand in ["PMI","pmi","post_mortem_interval","PostMortemInterval","PMI_h"]:
        if cand in obs.columns: return cand
    return None

def detect_sex_col(obs):
    for cand in ["Sex","sex","gender","Gender","Donor Sex","donor_sex"]:
        if cand in obs.columns: return cand
    return None

def donor_aggregate(obs, donor_col, cols):
    """Mean for numeric, first for categorical, indexed by donor."""
    g = obs.groupby(obs[donor_col].astype(str), dropna=False)
    out = pd.DataFrame(index=g.size().index)
    out["_n_cells"] = g.size().values
    for c in cols:
        if c not in obs.columns: continue
        if pd.api.types.is_numeric_dtype(obs[c]):
            out[c] = g[c].mean(numeric_only=True)
        else:
            out[c] = g[c].apply(lambda x: x.dropna().astype(str).value_counts().index[0]
                                if x.dropna().size else np.nan)
    return out.reset_index().rename(columns={"index": donor_col})

# Load MES gene sets from NB3 output (auto-discover Main_Table1.xlsx)
log("=" * 72)
log("LOAD: MES gene weights (auto-discovering)", tag="load")

def _find_gene_weights():
    """Locate the GeneWeights table without trusting a fixed path.

    NB3 saves it as Main_Table1.xlsx with sheet 'GeneWeights' (8 MES columns +
    'gene'). Search order:
      1. Likely path variants under BASE_DIR
      2. Recursive glob for *Table1*.xlsx under BASE_DIR
      3. Recursive glob for any .xlsx with a sheet named 'GeneWeights'
    Returns (path, df) or (None, None).
    """
    tried = []
    # 1) Path variants
    for ms_dir in ["Manuscript data", "Manuscript Data", "Manuscript_Data",
                   "manuscript_data", "manuscript data", "Manuscript",
                   "Process Data"]:
        for tab_dir in ["Tables", "tables", "Table", ""]:
            for fname in ["Main_Table1.xlsx", "Main_Table_1.xlsx",
                          "Table1.xlsx", "Main_Table1_GeneWeights.xlsx",
                          "MES_GeneWeights.xlsx"]:
                if tab_dir:
                    p = BASE_DIR / ms_dir / tab_dir / fname
                else:
                    p = BASE_DIR / ms_dir / fname
                tried.append(p)
                if p.exists():
                    try:
                        df = pd.read_excel(p, sheet_name="GeneWeights")
                        if any(str(c).startswith("MES") for c in df.columns):
                            return p, df
                    except Exception as e:
                        log(f"    found {p.name} but cannot read GeneWeights: {e}")

    # 2) Recursive glob for Table1-like filenames
    log("  fallback: recursive glob for *Table1*.xlsx under BASE_DIR")
    for p in BASE_DIR.rglob("*Table1*.xlsx"):
        tried.append(p)
        try:
            xls = pd.ExcelFile(p)
            if "GeneWeights" in xls.sheet_names:
                df = pd.read_excel(p, sheet_name="GeneWeights")
                if any(str(c).startswith("MES") for c in df.columns):
                    return p, df
        except Exception:
            continue

    # 3) Any xlsx with a GeneWeights sheet
    log("  fallback: scanning all xlsx under BASE_DIR for sheet 'GeneWeights'")
    for p in BASE_DIR.rglob("*.xlsx"):
        if "~$" in p.name:  # skip Excel lock files
            continue
        try:
            xls = pd.ExcelFile(p)
        except Exception:
            continue
        if "GeneWeights" in xls.sheet_names:
            try:
                df = pd.read_excel(p, sheet_name="GeneWeights")
                if any(str(c).startswith("MES") for c in df.columns):
                    return p, df
            except Exception:
                continue
    log(f"  exhausted {len(tried)} candidates. First 5 tried:")
    for p in tried[:5]:
        log(f"    {p}")
    return None, None

MAIN_T1, df_weights = _find_gene_weights()
if MAIN_T1 is None:
    raise FileNotFoundError(
        "Could not locate a Main_Table1.xlsx with a 'GeneWeights' sheet under "
        f"{BASE_DIR}. Either NB3 has not been run, or the file is in an "
        "unexpected location. Search the disk with `dir /s /b Main_Table1.xlsx` "
        "from a Windows shell and pass that path as MAIN_T1 directly."
    )
log(f"  found: {MAIN_T1}")

mes_cols_all = [c for c in df_weights.columns if str(c).startswith("MES")]
if len(mes_cols_all) != 8:
    log(f"  WARNING: expected 8 MES columns, found {len(mes_cols_all)}: {mes_cols_all}")

TOP_N_DEFAULT = 50
mes_gene_sets = {}
for mes in mes_cols_all:
    sub = df_weights[["gene", mes]].dropna().sort_values(mes, ascending=False).head(TOP_N_DEFAULT)
    mes_gene_sets[mes] = sub["gene"].astype(str).str.upper().tolist()

log(f"  loaded {len(mes_cols_all)} modules, top-{TOP_N_DEFAULT} genes each")

# Tolerance score panels
TOL_HOMEO = ["P2RY12", "CX3CR1", "TMEM119", "GPR34", "SALL1", "CSF1R", "OLFML3"]
TOL_ACTIV = ["APOE", "SPP1", "LPL", "TREM2", "CST7", "CTSD", "TYROBP", "FCER1G", "LGALS3", "CD68"]
TOL_ACTIV_NO_LGALS3 = [g for g in TOL_ACTIV if g != "LGALS3"]

GR_CORE = ["NR3C1", "FKBP5", "TSC22D3", "DDIT4", "KLF9"]
IRISIN  = ["FNDC5", "PPARGC1A", "PPRC1", "NRF1", "TFAM"]
MICRO_MARKERS = ["P2RY12", "CX3CR1", "TMEM119", "CSF1R", "AIF1", "TYROBP", "LST1"]

# Original (compact) axis priors as used in NB3
AXIS_ORIG = {
    "chemokine": ["CXCL12","CXCR4","CCL19","CCR7","CCL21","CXCL8","CXCL10","CXCR3",
                  "CCL5","CCR5","CXCL9","CXCL11","CCR2","CCL2"],
    "ecm_adhesion": ["FN1","ITGA5","ITGB1","ITGA4","ITGB7","LAMC1","LAMA4","LAMB1",
                     "COL1A1","COL1A2","COL3A1","VCAN","ICAM1","VCAM1","LGALS3"],
    "guidance": ["SEMA3A","SEMA3C","SEMA4D","SEMA7A","NRP1","NRP2","EPHA2","EPHA4",
                 "EPHB2","EFNA1","EFNA5","EFNB2","SLIT2","ROBO1","ROBO2"],
    "tolerance_assoc": ["FOXP3","IL2RA","CTLA4","IKZF2","TNFRSF18","LAG3","TIGIT","ICOS",
                        "TNFRSF4","IL10","TGFB1","BATF","IRF4"],
}
AXIS_ORIG = {k: [g.upper() for g in v] for k, v in AXIS_ORIG.items()}

# Expanded axis priors (~30-50 genes each) for R3.13
AXIS_EXPANDED = {
    "chemokine": [
        "CXCL12","CXCR4","CCL19","CCR7","CCL21","CXCL8","CXCL10","CXCR3","CCL5","CCR5",
        "CXCL9","CXCL11","CCR2","CCL2","CCL3","CCL4","CCL8","CCL13","CCL17","CCL20",
        "CCL22","CXCL1","CXCL2","CXCL3","CXCL5","CXCL6","CXCL13","CXCL14","CXCL16",
        "CCR1","CCR3","CCR4","CCR6","CCR8","CCR9","CCR10","CXCR1","CXCR2","CXCR5","CXCR6",
    ],
    "ecm_adhesion": [
        "FN1","ITGA5","ITGB1","ITGA4","ITGB7","LAMC1","LAMA4","LAMB1","LAMB2","LAMA2",
        "COL1A1","COL1A2","COL3A1","COL4A1","COL4A2","COL6A1","COL6A2","COL6A3",
        "VCAN","ICAM1","VCAM1","LGALS1","LGALS3","LGALS9","CD44","CD9","CD81",
        "SPARC","SPARCL1","DCN","MGP","BGN","TIMP1","TIMP2","TIMP3","MMP2","MMP9",
        "ITGAM","ITGAX","ITGB2","ITGAL","ITGA1","ITGA2","ITGA6","ITGA9",
    ],
    "guidance": [
        "SEMA3A","SEMA3B","SEMA3C","SEMA3D","SEMA3E","SEMA3F","SEMA4A","SEMA4B","SEMA4D",
        "SEMA6A","SEMA6B","SEMA6C","SEMA6D","SEMA7A",
        "NRP1","NRP2","PLXNA1","PLXNA2","PLXNA3","PLXNA4","PLXNB1","PLXNB2","PLXNB3",
        "PLXNC1","PLXND1","EPHA1","EPHA2","EPHA3","EPHA4","EPHA5","EPHA7",
        "EPHB1","EPHB2","EPHB3","EPHB4","EPHB6","EFNA1","EFNA2","EFNA3","EFNA4","EFNA5",
        "EFNB1","EFNB2","EFNB3","SLIT1","SLIT2","SLIT3","ROBO1","ROBO2","ROBO3","ROBO4",
        "DCC","UNC5A","UNC5B","UNC5C","UNC5D","NTN1","NTN3","NTN4",
    ],
    "tolerance_assoc": [
        "FOXP3","IL2RA","CTLA4","IKZF2","TNFRSF18","LAG3","TIGIT","HAVCR2","PDCD1",
        "ICOS","TNFRSF4","TNFRSF9","TNFRSF25","IL10","IL10RA","IL10RB",
        "TGFB1","TGFB2","TGFB3","TGFBR1","TGFBR2","BATF","IRF4","IKZF4","NRP1",
        "ENTPD1","NT5E","ARG1","ARG2","IDO1","IDO2","CD274","PDCD1LG2","BTLA","VTCN1",
        "LAIR1","CD83","CD86","HLA-DRA","HLA-DRB1","HLA-DPA1","HLA-DPB1","HLA-DQA1","HLA-DQB1",
        "AIRE","FEZF2","BCL2L1","SOCS1","SOCS3",
    ],
}
AXIS_EXPANDED = {k: [g.upper() for g in v] for k, v in AXIS_EXPANDED.items()}

mes_cols = mes_cols_all  # alias for brevity

# Load scored microglia cohorts
log("=" * 72)
log("LOAD: scored microglia cohorts", tag="load")
scored_files = sorted(glob.glob(str(AIM2_DIR / "*__microglia_scored.h5ad")))
cohorts = {}
for f in scored_files:
    ds = Path(f).name.replace("__microglia_scored.h5ad", "")
    try:
        a = sc.read_h5ad(f)
        if not a.obs_names.is_unique: a.obs_names_make_unique()
        if not a.var_names.is_unique: a.var_names_make_unique()
        cohorts[ds] = a
        log(f"  {ds}: {a.n_obs:,} cells × {a.n_vars:,} genes")
    except Exception as e:
        log(f"  FAILED {ds}: {e}")

if len(cohorts) == 0:
    log("  WARNING: no scored cohorts found. Some analyses will be skipped.")

# A1. MES-tolerance gene overlap (circularity diagnostic)
# Addresses: own issue #16 (Fig 2C LGALS3 is in the activation panel)
log("=" * 72)
log("A1: MES-tolerance gene overlap diagnostic", tag="A1")
try:
    homeo_set = {g.upper() for g in TOL_HOMEO}
    activ_set = {g.upper() for g in TOL_ACTIV}
    tol_all   = homeo_set | activ_set

    a1_rows = []
    for mes in mes_cols:
        top_set = set(mes_gene_sets[mes])
        ov_h = top_set & homeo_set
        ov_a = top_set & activ_set
        a1_rows.append({
            "module": mes,
            "top_N": len(top_set),
            "overlap_homeostatic": len(ov_h),
            "overlap_activation": len(ov_a),
            "overlap_total_tolscore": len(ov_h) + len(ov_a),
            "pct_top50_in_tolscore": round(100.0 * (len(ov_h) + len(ov_a)) / max(len(top_set), 1), 2),
            "homeo_genes": ", ".join(sorted(ov_h)),
            "activ_genes": ", ".join(sorted(ov_a)),
        })
    df_a1 = pd.DataFrame(a1_rows)
    save_xlsx({"MES_TolScore_Overlap": df_a1}, "A1_MES_TolScore_Overlap")
    log(f"  done. Max overlap pct: {df_a1['pct_top50_in_tolscore'].max():.1f}%")
    log(f"  Modules with overlap > 0: {(df_a1['overlap_total_tolscore'] > 0).sum()}/8")
except Exception as e:
    log(f"  A1 FAILED: {e}\n{traceback.format_exc()}")

# A2. Tolerance score without LGALS3 (circularity removal)
# Addresses: own #15 - LGALS3 is in activation panel AND was used as a result.
# Recompute tolerance_positioning and all MES correlations.
log("=" * 72)
log("A2: Tolerance score sans LGALS3 - recompute MES correlations", tag="A2")
try:
    a2_rows = []
    for ds, a in cohorts.items():
        prep_for_scoring(a)
        n_homeo = score_geneset(a, TOL_HOMEO, "score_homeostatic_v2", min_genes=3)
        n_act_full = score_geneset(a, TOL_ACTIV, "score_activation_v2", min_genes=3)
        n_act_noL  = score_geneset(a, TOL_ACTIV_NO_LGALS3, "score_activation_noLGALS3", min_genes=3)
        if not (n_homeo and n_act_full and n_act_noL):
            continue
        a.obs["tol_pos_v2"]       = a.obs["score_homeostatic_v2"] - a.obs["score_activation_v2"]
        a.obs["tol_pos_noLGALS3"] = a.obs["score_homeostatic_v2"] - a.obs["score_activation_noLGALS3"]

        donor_col = detect_donor_col(a.obs)
        score_cols = ["tol_pos_v2", "tol_pos_noLGALS3"] + [f"{m}_score" for m in mes_cols]
        score_cols = [c for c in score_cols if c in a.obs.columns]
        if donor_col is not None:
            ddf = donor_aggregate(a.obs, donor_col, score_cols)
            level = "donor"
        else:
            ddf = a.obs[score_cols].copy()
            level = "cell"

        for mes in mes_cols:
            mscol = f"{mes}_score"
            if mscol not in ddf.columns: continue
            r_orig = safe_corr_with_ci(ddf["tol_pos_v2"], ddf[mscol], method="spearman", n_boot=500)
            r_noL  = safe_corr_with_ci(ddf["tol_pos_noLGALS3"], ddf[mscol], method="spearman", n_boot=500)
            a2_rows.append({
                "dataset": ds, "level": level, "MES": mes,
                "N": r_orig["n"],
                "r_orig_tol": r_orig["r"], "ci_lo_orig": r_orig["ci_lo"], "ci_hi_orig": r_orig["ci_hi"],
                "r_noLGALS3":  r_noL["r"],  "ci_lo_noL":  r_noL["ci_lo"],  "ci_hi_noL":  r_noL["ci_hi"],
                "delta_r": (r_noL["r"] - r_orig["r"]) if np.isfinite(r_noL["r"]) and np.isfinite(r_orig["r"]) else np.nan,
            })

    df_a2 = pd.DataFrame(a2_rows)
    if len(df_a2):
        df_a2["abs_delta"] = df_a2["delta_r"].abs()
        med_delta = df_a2["abs_delta"].median()
        log(f"  median |Δρ| from LGALS3 removal: {med_delta:.4f}")
        save_xlsx({"MES_vs_Tol_withWithoutLGALS3": df_a2}, "A2_LGALS3_Sensitivity")
    else:
        log("  no rows produced (no scored cohorts had usable data)")
except Exception as e:
    log(f"  A2 FAILED: {e}\n{traceback.format_exc()}")

# A3. MES top-N cutoff sensitivity (R3.14)
# Rescore MES with top 25, 50, 100 genes by NMF weight.
# Recompute MES-tolerance correlations.
log("=" * 72)
log("A3: MES top-N cutoff sensitivity (N=25, 50, 100)", tag="A3")
try:
    top_n_variants = [25, 50, 100]
    a3_rows = []
    for ds, a in cohorts.items():
        prep_for_scoring(a)
        # tolerance score (using existing if present, else recompute)
        if "tolerance_positioning" not in a.obs.columns:
            score_geneset(a, TOL_HOMEO, "score_homeo_a3")
            score_geneset(a, TOL_ACTIV, "score_act_a3")
            a.obs["tolerance_positioning"] = a.obs["score_homeo_a3"] - a.obs["score_act_a3"]

        donor_col = detect_donor_col(a.obs)
        # build score columns for each (mes, topN)
        all_score_cols = ["tolerance_positioning"]
        for topN in top_n_variants:
            for mes in mes_cols:
                sub = df_weights[["gene", mes]].dropna().sort_values(mes, ascending=False).head(topN)
                genes = sub["gene"].astype(str).str.upper().tolist()
                col_name = f"{mes}_top{topN}"
                n_pres = score_geneset(a, genes, col_name, min_genes=3)
                if n_pres == 0: continue
                all_score_cols.append(col_name)

        score_cols_in_obs = [c for c in all_score_cols if c in a.obs.columns]
        if donor_col is not None:
            ddf = donor_aggregate(a.obs, donor_col, score_cols_in_obs)
            level = "donor"
        else:
            ddf = a.obs[score_cols_in_obs].copy()
            level = "cell"

        for topN in top_n_variants:
            for mes in mes_cols:
                col_name = f"{mes}_top{topN}"
                if col_name not in ddf.columns: continue
                r = safe_corr_with_ci(ddf["tolerance_positioning"], ddf[col_name],
                                       method="spearman", n_boot=500)
                a3_rows.append({
                    "dataset": ds, "level": level, "MES": mes, "top_N": topN,
                    "N": r["n"], "r": r["r"], "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"], "p": r["p"]
                })

    df_a3 = pd.DataFrame(a3_rows)
    if len(df_a3):
        df_a3["q_BH"] = bh_fdr(df_a3["p"].values)
        # pivot: per cohort, per MES, r at each topN
        pivot = df_a3.pivot_table(index=["dataset","MES"], columns="top_N", values="r", aggfunc="mean")
        if not pivot.empty:
            pivot["delta_25_vs_50"]  = pivot.get(25,  np.nan) - pivot.get(50, np.nan)
            pivot["delta_100_vs_50"] = pivot.get(100, np.nan) - pivot.get(50, np.nan)
        save_xlsx({"long_form": df_a3, "pivot_r_by_topN": pivot.reset_index()},
                  "A3_TopN_Sensitivity")
        med_d = float(pivot.get("delta_25_vs_50", pd.Series([np.nan])).abs().median()) if "delta_25_vs_50" in pivot.columns else np.nan
        log(f"  median |Δρ| top25 vs top50: {med_d:.4f}")
except Exception as e:
    log(f"  A3 FAILED: {e}\n{traceback.format_exc()}")

# A4. Expanded axis priors hypergeometric retest (R3.13, own #1)
# Original priors are ~14 genes per axis. With top-100 module size,
# k often == 1, producing identical fold-enrichment 1.801 etc.
# Use expanded priors (~30-50 genes each) and re-test.
log("=" * 72)
log("A4: Expanded axis priors hypergeometric retest", tag="A4")
try:
    # Universe = unique union of MES-weight gene set + expanded priors
    universe = set(df_weights["gene"].astype(str).str.upper().tolist())
    for s in AXIS_EXPANDED.values():
        universe |= set(s)
    M = len(universe)

    def hyper_test(module_genes, axis_genes, M):
        mset = set(g.upper() for g in module_genes)
        aset = set(axis_genes) & universe
        overlap = mset & aset
        k = len(overlap)
        N = len(mset & universe)
        n = len(aset)
        if k == 0 or n == 0 or N == 0:
            return {"overlap": k, "module_size": N, "axis_size": n,
                    "fold": 0.0, "pval": 1.0, "overlap_genes": ""}
        pval = float(stats.hypergeom.sf(k - 1, M, n, N))
        fold = (k / N) / (n / M)
        return {"overlap": k, "module_size": N, "axis_size": n,
                "fold": round(float(fold), 3), "pval": pval,
                "overlap_genes": ", ".join(sorted(overlap))}

    a4_rows = []
    for mes in mes_cols:
        top100 = df_weights[["gene", mes]].dropna().sort_values(mes, ascending=False).head(100)
        top100_genes = top100["gene"].astype(str).str.upper().tolist()
        # original priors result (for comparison)
        for ax_name, ax_genes in AXIS_ORIG.items():
            r = hyper_test(top100_genes, ax_genes, M)
            r.update({"module": mes, "axis": ax_name, "prior_set": "original"})
            a4_rows.append(r)
        for ax_name, ax_genes in AXIS_EXPANDED.items():
            r = hyper_test(top100_genes, ax_genes, M)
            r.update({"module": mes, "axis": ax_name, "prior_set": "expanded"})
            a4_rows.append(r)
    df_a4 = pd.DataFrame(a4_rows)
    # Apply BH within each prior_set (original / expanded), across all module x axis
    for ps in df_a4["prior_set"].unique():
        mask = df_a4["prior_set"] == ps
        df_a4.loc[mask, "pval_BH"] = bh_fdr(df_a4.loc[mask, "pval"].values)
    df_a4["sig_BH005"] = df_a4["pval_BH"] < 0.05

    # Per module: best axis under expanded priors
    best_rows = []
    for mes in mes_cols:
        sub = df_a4[(df_a4["module"] == mes) & (df_a4["prior_set"] == "expanded")]
        if len(sub) == 0: continue
        srt = sub.sort_values("fold", ascending=False).iloc[0]
        best_rows.append({
            "module": mes,
            "best_axis_expanded": srt["axis"],
            "fold_expanded": srt["fold"],
            "pval_BH_expanded": srt["pval_BH"],
            "overlap_n": srt["overlap"],
            "overlap_genes": srt["overlap_genes"],
        })
    df_a4_best = pd.DataFrame(best_rows)

    save_xlsx({"full_long": df_a4, "best_axis_expanded": df_a4_best}, "A4_AxisPriors_Expanded")
    n_sig = int((df_a4["prior_set"] == "expanded").sum() and df_a4.loc[df_a4["prior_set"]=="expanded","sig_BH005"].sum())
    log(f"  expanded priors: {n_sig} (module x axis) combinations reach BH q<0.05")
except Exception as e:
    log(f"  A4 FAILED: {e}\n{traceback.format_exc()}")

# A5. NMF factor cross-correlation matrix (R3.12)
# H_ts row vectors (one per module, length = n_genes in universe).
# Pairwise Pearson across modules; off-diagonal magnitudes report redundancy.
log("=" * 72)
log("A5: NMF factor (H_ts) cross-correlation matrix", tag="A5")
try:
    # H_ts ≈ GeneWeights.T (genes × modules) -> modules × genes
    gw = df_weights.dropna(subset=["gene"]).copy()
    gw["gene"] = gw["gene"].astype(str).str.upper()
    H_ts = gw[mes_cols].fillna(0.0).to_numpy().T   # shape (8, n_genes)
    # z-score each row before correlation to handle scale differences
    Hz = (H_ts - H_ts.mean(axis=1, keepdims=True)) / (H_ts.std(axis=1, keepdims=True) + 1e-12)
    C = Hz @ Hz.T / Hz.shape[1]
    off_diag = C[np.triu_indices(C.shape[0], k=1)]
    log(f"  off-diagonal: max |r|={np.max(np.abs(off_diag)):.3f}, "
        f"median |r|={np.median(np.abs(off_diag)):.3f}")
    # heatmap
    fig, ax = plt.subplots(figsize=(5.5, 4.6))
    if HAS_SNS:
        sns.heatmap(C, ax=ax, cmap="RdBu_r", vmin=-1, vmax=1,
                    annot=True, fmt=".2f", annot_kws={"fontsize":6},
                    xticklabels=mes_cols, yticklabels=mes_cols,
                    cbar_kws={"label":"Pearson r","shrink":0.8})
    else:
        im = ax.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
        ax.set_xticks(range(8)); ax.set_xticklabels(mes_cols, rotation=45, ha="right")
        ax.set_yticks(range(8)); ax.set_yticklabels(mes_cols)
        plt.colorbar(im, ax=ax, label="Pearson r")
    ax.set_title("Inter-module correlation (H_ts rows, z-scored)")
    save_fig(fig, "A5_NMF_FactorCrossCorr")

    df_a5 = pd.DataFrame(C, index=mes_cols, columns=mes_cols).reset_index().rename(
        columns={"index":"module"})
    summary = pd.DataFrame([{
        "max_abs_off_diag": float(np.max(np.abs(off_diag))),
        "median_abs_off_diag": float(np.median(np.abs(off_diag))),
        "pct_abs_gt_0.3": float(np.mean(np.abs(off_diag) > 0.3) * 100),
        "pct_abs_gt_0.5": float(np.mean(np.abs(off_diag) > 0.5) * 100),
    }])
    save_xlsx({"corr_matrix": df_a5, "summary": summary}, "A5_NMF_FactorCrossCorr")
except Exception as e:
    log(f"  A5 FAILED: {e}\n{traceback.format_exc()}")

# A6. Cohort gene-coverage report (R3.13)
# For each validation cohort, what fraction of each MES top-50 is present?
log("=" * 72)
log("A6: Per-cohort MES gene coverage", tag="A6")
try:
    a6_rows = []
    for ds, a in cohorts.items():
        var_set = {str(v).upper() for v in a.var_names}
        for mes in mes_cols:
            genes = set(mes_gene_sets[mes])
            present = genes & var_set
            a6_rows.append({
                "dataset": ds, "MES": mes,
                "n_top50": len(genes),
                "n_present": len(present),
                "pct_present": round(100.0 * len(present) / max(len(genes), 1), 1),
                "missing_genes": ", ".join(sorted(genes - var_set))[:300],
            })
    df_a6 = pd.DataFrame(a6_rows)
    save_xlsx({"GeneCoverage": df_a6}, "A6_GeneCoverage_PerCohort")
    pmin = df_a6["pct_present"].min() if len(df_a6) else np.nan
    pmean = df_a6["pct_present"].mean() if len(df_a6) else np.nan
    log(f"  coverage range: min {pmin:.1f}%, mean {pmean:.1f}%")
except Exception as e:
    log(f"  A6 FAILED: {e}\n{traceback.format_exc()}")

# A7. cNMF parameter dump for Methods (R3.11)
log("=" * 72)
log("A7: cNMF parameter dump", tag="A7")
try:
    cnmf_params = pd.DataFrame([
        {"parameter":"K_chosen","value":8},
        {"parameter":"K_sweep","value":"[5, 8, 10, 12]"},
        {"parameter":"consensus_runs_per_K","value":20},
        {"parameter":"within_TS_stability_seeds","value":10},
        {"parameter":"NMF_init","value":"nndsvda"},
        {"parameter":"NMF_solver","value":"sklearn default (CD)"},
        {"parameter":"max_iter","value":800},
        {"parameter":"base_random_state","value":42},
        {"parameter":"regularization","value":"none (alpha=0)"},
        {"parameter":"connectivity_distance","value":"1 - consensus_matrix"},
        {"parameter":"linkage_method","value":"average"},
        {"parameter":"cophenetic_metric","value":"squareform of (1-C), then cophenet(Z,d)"},
        {"parameter":"silhouette_metric","value":"cosine, sample 10000 cells per K"},
        {"parameter":"external_validation_seed","value":42},
        {"parameter":"external_validation_subsample","value":"30,000 cells per cohort"},
        {"parameter":"TS_training_subsample","value":"60,000 cells"},
        {"parameter":"matching_algorithm","value":"Hungarian (linear_sum_assignment) on Pearson"},
        {"parameter":"permutation_test_iters","value":500},
        {"parameter":"convergence_criterion","value":"sklearn default tol=1e-4"},
        {"parameter":"gene_universe","value":"HVG (Seurat v3 vst, 2500) union axis priors present"},
    ])
    save_xlsx({"cNMF_parameters": cnmf_params}, "A7_cNMF_Parameters")
except Exception as e:
    log(f"  A7 FAILED: {e}\n{traceback.format_exc()}")

# A8. HVG count sensitivity sweep (R3.9)  [SLOW]
# Retrain K=8 NMF at HVG=1500, 2500, 3500. Hungarian-match against 2500.
# Report per-module matched correlation.
def _X_dense(a, genes):
    X = a[:, genes].X
    return X.tocsr() if sp.issparse(X) else sp.csr_matrix(X)

def _subsample(n_obs, fit_n, rng):
    if n_obs > fit_n:
        return rng.choice(n_obs, size=fit_n, replace=False)
    return np.arange(n_obs)

def corr_components(H_a, H_b):
    A = (H_a - H_a.mean(axis=1, keepdims=True)) / (H_a.std(axis=1, keepdims=True) + 1e-12)
    B = (H_b - H_b.mean(axis=1, keepdims=True)) / (H_b.std(axis=1, keepdims=True) + 1e-12)
    return (A @ B.T) / A.shape[1]

def hungarian_match(C):
    r, c = linear_sum_assignment(-C)
    return r, c, C[r, c]

if RUN_SLOW_NMF:
    log("=" * 72)
    log("A8: HVG count sensitivity sweep (NMF retraining, may be slow)", tag="A8")
    try:
        if TS_QC is None or not Path(TS_QC).exists():
            raise FileNotFoundError("TS_Thymus_filtered__qc.h5ad not found; skipping A8")
        ts = sc.read_h5ad(TS_QC)
        ensure_counts(ts)
        ts_train_base = ts.copy(); ts_train_base.X = ts_train_base.layers["counts"].copy()
        sc.pp.normalize_total(ts_train_base, target_sum=1e4); sc.pp.log1p(ts_train_base)

        axis_all_orig = {g for v in AXIS_ORIG.values() for g in v}
        results_by_hvg = {}
        rng = np.random.RandomState(SEED)
        FIT_N = 60000
        idx_fit = _subsample(ts_train_base.n_obs, FIT_N, rng)

        for hvg_n in [1500, 2500, 3500]:
            log(f"  HVG={hvg_n}: selecting HVG and training K=8 NMF")
            tmp = ts_train_base.copy()
            sc.pp.highly_variable_genes(tmp, n_top_genes=hvg_n, flavor="seurat_v3")
            hvg = tmp.var_names[tmp.var["highly_variable"]].tolist()
            present_axis = [v for v in tmp.var_names
                            if str(v).upper() in axis_all_orig and v not in hvg]
            universe = list(dict.fromkeys(hvg + present_axis))
            ts_fit = tmp[idx_fit, universe].copy()
            Xfit = _X_dense(ts_fit, universe)
            model = NMF(n_components=8, init="nndsvda", random_state=SEED, max_iter=800)
            model.fit(Xfit)
            H = model.components_
            results_by_hvg[hvg_n] = {"H": H, "universe": [g.upper() for g in universe]}

        # Match against HVG=2500 as reference
        ref = results_by_hvg[2500]
        a8_rows = []
        for hvg_n in [1500, 3500]:
            tgt = results_by_hvg[hvg_n]
            shared = sorted(set(ref["universe"]) & set(tgt["universe"]))
            pos_ref = {g: i for i, g in enumerate(ref["universe"])}
            pos_tgt = {g: i for i, g in enumerate(tgt["universe"])}
            cols_ref = [pos_ref[g] for g in shared]
            cols_tgt = [pos_tgt[g] for g in shared]
            H_ref_s = ref["H"][:, cols_ref]
            H_tgt_s = tgt["H"][:, cols_tgt]
            C = corr_components(H_ref_s, H_tgt_s)
            r, c, vals = hungarian_match(C)
            for k_ref, k_tgt, v in zip(r, c, vals):
                a8_rows.append({
                    "ref_HVG": 2500, "tgt_HVG": hvg_n,
                    "ref_module": f"MES{k_ref+1:02d}",
                    "tgt_module": f"MES{k_tgt+1:02d}",
                    "matched_corr": float(v),
                    "n_shared_genes": len(shared),
                })
        df_a8 = pd.DataFrame(a8_rows)
        if len(df_a8):
            summary_a8 = df_a8.groupby("tgt_HVG").agg(
                mean_match=("matched_corr","mean"),
                median_match=("matched_corr","median"),
                min_match=("matched_corr","min")
            ).reset_index()
            save_xlsx({"per_module": df_a8, "summary": summary_a8}, "A8_HVG_Sensitivity")
            log(f"  mean matched corr: HVG1500={summary_a8.loc[summary_a8['tgt_HVG']==1500,'mean_match'].iloc[0]:.3f}, "
                f"HVG3500={summary_a8.loc[summary_a8['tgt_HVG']==3500,'mean_match'].iloc[0]:.3f}")

        # Save H_ts at HVG=2500 (reference, for downstream A9 use)
        results_2500 = results_by_hvg[2500]
        del ts_train_base, ts; gc.collect()
    except Exception as e:
        log(f"  A8 FAILED: {e}\n{traceback.format_exc()}")
        results_by_hvg = None
else:
    log("A8 skipped (RUN_SLOW_NMF=False)", tag="A8")
    results_by_hvg = None

# A9. NMF without curated gene priors (R3.8)  [SLOW]
# Train K=8 NMF on HVG-only (no axis_present supplement).
# Hungarian-match against original (HVG + axis_present).
if RUN_SLOW_NMF:
    log("=" * 72)
    log("A9: NMF without curated gene priors (HVG-only)", tag="A9")
    try:
        if TS_QC is None or not Path(TS_QC).exists():
            raise FileNotFoundError("TS_Thymus_filtered__qc.h5ad not found; skipping A9")
        ts = sc.read_h5ad(TS_QC)
        ensure_counts(ts)
        ts_train = ts.copy(); ts_train.X = ts_train.layers["counts"].copy()
        sc.pp.normalize_total(ts_train, target_sum=1e4); sc.pp.log1p(ts_train)
        sc.pp.highly_variable_genes(ts_train, n_top_genes=2500, flavor="seurat_v3")

        hvg = ts_train.var_names[ts_train.var["highly_variable"]].tolist()
        axis_all = {g for v in AXIS_ORIG.values() for g in v}
        axis_present = [v for v in ts_train.var_names if str(v).upper() in axis_all and v not in hvg]
        universe_orig = list(dict.fromkeys(hvg + axis_present))
        universe_noprior = hvg[:]

        rng = np.random.RandomState(SEED)
        idx_fit = _subsample(ts_train.n_obs, 60000, rng)

        # Original
        ts_orig = ts_train[idx_fit, universe_orig].copy()
        m_orig = NMF(n_components=8, init="nndsvda", random_state=SEED, max_iter=800)
        m_orig.fit(_X_dense(ts_orig, universe_orig))
        H_orig = m_orig.components_

        # No prior
        ts_np = ts_train[idx_fit, universe_noprior].copy()
        m_np = NMF(n_components=8, init="nndsvda", random_state=SEED, max_iter=800)
        m_np.fit(_X_dense(ts_np, universe_noprior))
        H_np = m_np.components_

        shared = sorted(set(g.upper() for g in universe_orig) & set(g.upper() for g in universe_noprior))
        po = {g.upper(): i for i, g in enumerate(universe_orig)}
        pn = {g.upper(): i for i, g in enumerate(universe_noprior)}
        H_orig_s = H_orig[:, [po[g] for g in shared]]
        H_np_s   = H_np[:,   [pn[g] for g in shared]]
        C = corr_components(H_orig_s, H_np_s)
        r, c, vals = hungarian_match(C)

        a9_rows = []
        for k_o, k_n, v in zip(r, c, vals):
            a9_rows.append({
                "orig_module": f"MES{k_o+1:02d}",
                "noprior_module": f"MES{k_n+1:02d}",
                "matched_corr": float(v),
            })
        df_a9 = pd.DataFrame(a9_rows)
        a9_summary = pd.DataFrame([{
            "n_modules": 8,
            "mean_match": float(df_a9["matched_corr"].mean()),
            "median_match": float(df_a9["matched_corr"].median()),
            "min_match": float(df_a9["matched_corr"].min()),
            "n_modules_match_gt_0.7": int((df_a9["matched_corr"] > 0.7).sum()),
        }])
        save_xlsx({"per_module": df_a9, "summary": a9_summary}, "A9_NoPriors")
        log(f"  no-prior vs orig: mean matched r = {df_a9['matched_corr'].mean():.3f}, "
            f"min = {df_a9['matched_corr'].min():.3f}")

        del ts, ts_train, ts_orig, ts_np, m_orig, m_np; gc.collect()
    except Exception as e:
        log(f"  A9 FAILED: {e}\n{traceback.format_exc()}")
else:
    log("A9 skipped (RUN_SLOW_NMF=False)", tag="A9")

# A10. Microglia purity per cohort (R3.7)
# Score MICRO_MARKERS panel per cell, report purity distributions,
# stratify MES-tolerance correlation by purity quartile (focus: Tuddenham).
log("=" * 72)
log("A10: Microglia purity per cohort + stratified correlations", tag="A10")
try:
    a10_dist = []
    a10_strat = []
    for ds, a in cohorts.items():
        prep_for_scoring(a)
        n_pres = score_geneset(a, MICRO_MARKERS, "_microglia_purity", min_genes=3)
        if n_pres == 0:
            log(f"  {ds}: insufficient microglia markers ({n_pres}), skipping")
            continue
        pur = pd.to_numeric(a.obs["_microglia_purity"], errors="coerce").to_numpy()
        finite = np.isfinite(pur)
        a10_dist.append({
            "dataset": ds, "n_cells": int(finite.sum()),
            "purity_min": float(np.nanmin(pur)),
            "purity_q25": float(np.nanpercentile(pur, 25)),
            "purity_median": float(np.nanmedian(pur)),
            "purity_q75": float(np.nanpercentile(pur, 75)),
            "purity_max": float(np.nanmax(pur)),
            "n_markers_used": n_pres,
        })

        # Quartile stratification (cell-level, since donor info often weak in Tuddenham/MS)
        if "tolerance_positioning" not in a.obs.columns:
            score_geneset(a, TOL_HOMEO, "_homeo_a10"); score_geneset(a, TOL_ACTIV, "_act_a10")
            a.obs["tolerance_positioning"] = a.obs["_homeo_a10"] - a.obs["_act_a10"]

        qmask = ["Q1 (low)", "Q2", "Q3", "Q4 (high)"]
        bounds = np.nanpercentile(pur, [0, 25, 50, 75, 100])
        for qi, label in enumerate(qmask):
            lo, hi = bounds[qi], bounds[qi+1]
            sel = (pur >= lo) & (pur <= hi)
            if sel.sum() < 50: continue
            sub = a.obs.loc[sel]
            for mes in mes_cols:
                msc = f"{mes}_score"
                if msc not in sub.columns: continue
                r = safe_corr_with_ci(sub["tolerance_positioning"], sub[msc],
                                       method="spearman", n_boot=300)
                a10_strat.append({
                    "dataset": ds, "purity_quartile": label,
                    "purity_lo": float(lo), "purity_hi": float(hi),
                    "n_cells": int(sel.sum()),
                    "MES": mes, "r": r["r"], "p": r["p"],
                    "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"],
                })

    df_a10_dist = pd.DataFrame(a10_dist)
    df_a10_strat = pd.DataFrame(a10_strat)
    save_xlsx({"PurityDistribution": df_a10_dist,
               "MES_vs_Tol_by_Purity": df_a10_strat}, "A10_Microglia_Purity")
    if "Tuddenham_GSE204702" in [r["dataset"] for r in a10_dist]:
        tud = df_a10_dist[df_a10_dist["dataset"] == "Tuddenham_GSE204702"]
        log(f"  Tuddenham purity median: {tud['purity_median'].iloc[0]:.3f}")
except Exception as e:
    log(f"  A10 FAILED: {e}\n{traceback.format_exc()}")

# A11. QC mito threshold sensitivity (R3.16)
# Re-filter scored h5ads on pct_counts_mt at 20/25/30/35%, recompute
# MES-tolerance Spearman per cohort. If pct_counts_mt missing, recompute it.
log("=" * 72)
log("A11: QC mito threshold sensitivity (20/25/30/35%)", tag="A11")
try:
    a11_rows = []
    for ds, a in cohorts.items():
        # Make sure pct_counts_mt exists
        if "pct_counts_mt" not in a.obs.columns:
            try:
                a.var["mt"] = a.var_names.astype(str).str.upper().str.startswith("MT-")
                sc.pp.calculate_qc_metrics(a, qc_vars=["mt"], percent_top=None,
                                            log1p=False, inplace=True)
            except Exception:
                log(f"  {ds}: cannot compute pct_counts_mt, skipping")
                continue
        if "tolerance_positioning" not in a.obs.columns:
            prep_for_scoring(a)
            score_geneset(a, TOL_HOMEO, "_h_a11"); score_geneset(a, TOL_ACTIV, "_a_a11")
            a.obs["tolerance_positioning"] = a.obs["_h_a11"] - a.obs["_a_a11"]

        donor_col = detect_donor_col(a.obs)
        mt_pct = pd.to_numeric(a.obs["pct_counts_mt"], errors="coerce").to_numpy()
        for thr in [20, 25, 30, 35]:
            keep = (mt_pct <= thr) & np.isfinite(mt_pct)
            if keep.sum() < 100: continue
            sub_obs = a.obs.loc[keep].copy()
            score_cols = ["tolerance_positioning"] + [f"{m}_score" for m in mes_cols]
            score_cols = [c for c in score_cols if c in sub_obs.columns]
            if donor_col is not None and donor_col in sub_obs.columns:
                ddf = donor_aggregate(sub_obs, donor_col, score_cols)
                level = "donor"
            else:
                ddf = sub_obs[score_cols].copy()
                level = "cell"
            for mes in mes_cols:
                msc = f"{mes}_score"
                if msc not in ddf.columns: continue
                r = safe_corr_with_ci(ddf["tolerance_positioning"], ddf[msc],
                                       method="spearman", n_boot=300)
                a11_rows.append({
                    "dataset": ds, "mito_thresh_pct": thr, "level": level,
                    "n_cells_kept": int(keep.sum()), "N_unit": r["n"],
                    "MES": mes, "r": r["r"], "p": r["p"],
                    "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"],
                })

    df_a11 = pd.DataFrame(a11_rows)
    if len(df_a11):
        # delta vs 20% (paper default)
        pivot = df_a11.pivot_table(index=["dataset","MES"], columns="mito_thresh_pct",
                                    values="r", aggfunc="mean").reset_index()
        for col in [25, 30, 35]:
            if col in pivot.columns and 20 in pivot.columns:
                pivot[f"delta_vs_20_{col}"] = pivot[col] - pivot[20]
        save_xlsx({"long": df_a11, "pivot": pivot}, "A11_Mito_Threshold_Sensitivity")
        med_d35 = pivot.get("delta_vs_20_35", pd.Series([np.nan])).abs().median()
        log(f"  median |Δρ| 35% vs 20%: {med_d35:.4f}")
except Exception as e:
    log(f"  A11 FAILED: {e}\n{traceback.format_exc()}")

# A12. Visium microglia-weighted MES-tolerance correlation (R1.2, R3.7)
# Reload Visium, score microglia panel and MES, compute:
#   (i) per-spot weighted Pearson with microglia score as weight
#   (ii) restrict to top-30% microglia-enriched spots, recompute correlation
# Compare to unweighted (paper default).
if RUN_VISIUM:
    log("=" * 72)
    log("A12: Visium microglia-weighted correlations", tag="A12")
    try:
        vis_root = None
        for p in RAW_DIR.rglob("GSE220442"):
            cand = p / "counts_and_images"
            if cand.exists():
                vis_root = cand; break
        assert vis_root is not None, "Visium counts_and_images not found"

        a12_rows = []
        samples = sorted([d for d in vis_root.iterdir() if d.is_dir()])
        log(f"  found {len(samples)} Visium samples")

        for sdir in samples:
            h5 = sdir / "filtered_feature_bc_matrix.h5"
            if not h5.exists(): continue
            try:
                a = sc.read_10x_h5(h5)
                a.var_names_make_unique()
                # ensure gene symbol convention
                a.var_names = pd.Index([str(v).upper() for v in a.var_names])
                a.var_names_make_unique()
                a.layers["counts"] = a.X.copy()
                sc.pp.normalize_total(a, target_sum=1e4)
                sc.pp.log1p(a)

                score_geneset(a, MICRO_MARKERS, "microglia_score", min_genes=3)
                score_geneset(a, TOL_HOMEO, "homeo_score", min_genes=3)
                score_geneset(a, TOL_ACTIV, "activ_score", min_genes=3)
                a.obs["tolerance_positioning"] = a.obs["homeo_score"] - a.obs["activ_score"]
                for mes in mes_cols:
                    score_geneset(a, mes_gene_sets[mes], f"{mes}_score", min_genes=3)

                tol = pd.to_numeric(a.obs["tolerance_positioning"], errors="coerce").to_numpy()
                mw  = pd.to_numeric(a.obs["microglia_score"], errors="coerce").to_numpy()
                # Threshold weight at 0 (negative microglia score = no microglia signal)
                w_pos = np.where((mw > 0) & np.isfinite(mw), mw, 0.0)
                # top 30% microglia spots
                mw_finite = mw[np.isfinite(mw)]
                if len(mw_finite) < 50: continue
                thr_top = np.nanpercentile(mw_finite, 70)
                top_mask = (mw >= thr_top) & np.isfinite(mw)

                for mes in mes_cols:
                    mscol = f"{mes}_score"
                    if mscol not in a.obs.columns: continue
                    msc = pd.to_numeric(a.obs[mscol], errors="coerce").to_numpy()
                    # unweighted (paper default)
                    r_u = safe_corr_with_ci(msc, tol, method="spearman", n_boot=300)
                    # weighted Pearson
                    r_w, p_w, n_eff = weighted_pearson(msc, tol, w_pos)
                    # top-30% microglia-enriched
                    if top_mask.sum() >= 20:
                        r_t = safe_corr_with_ci(msc[top_mask], tol[top_mask],
                                                method="spearman", n_boot=300)
                    else:
                        r_t = {"r": np.nan, "p": np.nan, "n": int(top_mask.sum()),
                               "ci_lo": np.nan, "ci_hi": np.nan}

                    a12_rows.append({
                        "sample": sdir.name, "MES": mes,
                        "n_spots_total": int(a.n_obs),
                        "r_unweighted": r_u["r"], "p_u": r_u["p"],
                        "ci_lo_u": r_u["ci_lo"], "ci_hi_u": r_u["ci_hi"],
                        "r_weighted_pearson": r_w, "p_w": p_w, "n_eff": n_eff,
                        "r_top30pct_microglia": r_t["r"],
                        "n_top30": int(top_mask.sum()), "p_t": r_t["p"],
                        "ci_lo_t": r_t["ci_lo"], "ci_hi_t": r_t["ci_hi"],
                    })
                log(f"  {sdir.name}: processed")
                del a; gc.collect()
            except Exception as e:
                log(f"  {sdir.name} FAILED: {e}")
                continue

        df_a12 = pd.DataFrame(a12_rows)
        if len(df_a12):
            df_a12["q_BH_unweighted"] = bh_fdr(df_a12["p_u"].values)
            df_a12["q_BH_weighted"]   = bh_fdr(df_a12["p_w"].values)
            df_a12["q_BH_top30"]      = bh_fdr(df_a12["p_t"].values)
            # Comparison heatmap: mean r per (sample x MES) across three methods
            piv_u = df_a12.pivot_table(index="sample", columns="MES", values="r_unweighted")
            piv_w = df_a12.pivot_table(index="sample", columns="MES", values="r_weighted_pearson")
            piv_t = df_a12.pivot_table(index="sample", columns="MES", values="r_top30pct_microglia")

            fig, axes = plt.subplots(1, 3, figsize=(13.5, max(2.4, 0.4 * piv_u.shape[0])),
                                      sharey=True, constrained_layout=True)
            for ax, mat, title in [(axes[0], piv_u, "Unweighted (paper default)"),
                                   (axes[1], piv_w, "Microglia-weighted Pearson"),
                                   (axes[2], piv_t, "Top-30% microglia spots")]:
                if HAS_SNS:
                    sns.heatmap(mat, ax=ax, cmap="RdBu_r", vmin=-0.5, vmax=0.5,
                                annot=True, fmt=".2f", annot_kws={"fontsize":6},
                                linewidths=0.4, linecolor="white",
                                cbar_kws={"label":"ρ","shrink":0.7})
                else:
                    im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-0.5, vmax=0.5)
                    ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels(mat.columns, rotation=45)
                    ax.set_yticks(range(mat.shape[0])); ax.set_yticklabels(mat.index)
                ax.set_title(title)
            save_fig(fig, "A12_Visium_Weighting_Comparison")
            save_xlsx({"long": df_a12, "pivot_unweighted": piv_u.reset_index(),
                       "pivot_weighted": piv_w.reset_index(),
                       "pivot_top30": piv_t.reset_index()},
                      "A12_Visium_MicrogliaWeighted")
            # Headline: mean MES02 (ECM/adhesion) across methods
            m02 = df_a12[df_a12["MES"] == "MES02"]
            if len(m02):
                log(f"  MES02 mean r: unweighted={m02['r_unweighted'].mean():.3f}, "
                    f"weighted={m02['r_weighted_pearson'].mean():.3f}, "
                    f"top30%={m02['r_top30pct_microglia'].mean():.3f}")
    except Exception as e:
        log(f"  A12 FAILED: {e}\n{traceback.format_exc()}")
else:
    log("A12 skipped (RUN_VISIUM=False)", tag="A12")

# A13. PMI / sex stratified sensitivity for GR-MES (R3.18)
# SEA-AD only (richest metadata).
log("=" * 72)
log("A13: PMI / sex stratified sensitivity (SEA-AD)", tag="A13")
try:
    if "SEA-AD" not in cohorts:
        log("  SEA-AD not loaded; skipping")
    else:
        a = cohorts["SEA-AD"]
        donor_col = detect_donor_col(a.obs)
        sex_col = detect_sex_col(a.obs)
        pmi_col = detect_pmi_col(a.obs)
        gr_col  = "GR_composite" if "GR_composite" in a.obs.columns else None

        a13_rows = []
        if donor_col is None or gr_col is None:
            log(f"  missing donor_col ({donor_col}) or GR_composite; skipping")
        else:
            score_cols = ["tolerance_positioning", gr_col] + [f"{m}_score" for m in mes_cols]
            score_cols = [c for c in score_cols if c in a.obs.columns]
            extras = [c for c in [sex_col, pmi_col] if c]
            cols_to_agg = score_cols + extras
            ddf = donor_aggregate(a.obs, donor_col, cols_to_agg)

            # Sex strata
            if sex_col and sex_col in ddf.columns:
                for sx in ddf[sex_col].dropna().astype(str).unique():
                    sub = ddf[ddf[sex_col].astype(str) == sx]
                    if len(sub) < 8: continue
                    for mes in mes_cols:
                        msc = f"{mes}_score"
                        if msc not in sub.columns: continue
                        r = safe_corr_with_ci(sub[gr_col], sub[msc], method="spearman", n_boot=500)
                        a13_rows.append({"stratum": "sex", "group": sx, "n_donors": len(sub),
                                         "MES": mes, "r": r["r"], "p": r["p"],
                                         "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"]})
            # PMI tertiles
            if pmi_col and pmi_col in ddf.columns:
                pmi_num = pd.to_numeric(ddf[pmi_col], errors="coerce")
                fin = pmi_num.dropna()
                if len(fin) >= 24:
                    t_lo, t_hi = fin.quantile([1/3, 2/3])
                    bins = {"PMI_low": pmi_num <= t_lo,
                            "PMI_mid": (pmi_num > t_lo) & (pmi_num <= t_hi),
                            "PMI_high": pmi_num > t_hi}
                    for label, mask in bins.items():
                        sub = ddf[mask]
                        if len(sub) < 8: continue
                        for mes in mes_cols:
                            msc = f"{mes}_score"
                            if msc not in sub.columns: continue
                            r = safe_corr_with_ci(sub[gr_col], sub[msc], method="spearman", n_boot=500)
                            a13_rows.append({"stratum": "PMI_tertile", "group": label,
                                             "n_donors": len(sub),
                                             "MES": mes, "r": r["r"], "p": r["p"],
                                             "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"]})

        df_a13 = pd.DataFrame(a13_rows)
        if len(df_a13):
            df_a13["q_BH"] = bh_fdr(df_a13["p"].values)
            save_xlsx({"GR_MES_stratified": df_a13}, "A13_PMI_Sex_Sensitivity")
            log(f"  {len(df_a13)} stratified correlations computed")
except Exception as e:
    log(f"  A13 FAILED: {e}\n{traceback.format_exc()}")

# A14. GR signature alternative derivations (own #17)
# Volcano in paper shows only ~1 gene passing strict cutoff.
# Three alternative signatures:
#   (a) curated core only (NR3C1, FKBP5, TSC22D3, DDIT4, KLF9)
#   (b) top by absolute Welch t-stat (no padj filter), top 30
#   (c) top by absolute log2FC (any padj), top 50
# Score each on GSE219208 ctrl/treated and compute AUC.
log("=" * 72)
log("A14: GR signature alternative derivations", tag="A14")
try:
    stress_hits = list(RAW_DIR.rglob("GSE219208_Non-Normalized_read_counts_combined_lanes_.csv"))
    if not stress_hits:
        log("  GSE219208 not found; skipping")
    else:
        df_stress_raw = pd.read_csv(stress_hits[0], index_col=0)

        def parse_grp(col):
            base = str(col).split("_")[0].lower()
            wash = "wo" in base
            if base.startswith("ctl") or base.startswith("cv") or base.startswith("dv"):
                return "Control" if not wash else "Washout"
            if base.startswith("cort") or base.startswith("dex"):
                return "Washout" if wash else "Treated"
            return "Unknown"

        meta = pd.DataFrame({"sample": df_stress_raw.columns})
        meta["group"] = meta["sample"].apply(parse_grp)
        ctrl = meta[meta["group"] == "Control"]["sample"].tolist()
        treat = meta[meta["group"] == "Treated"]["sample"].tolist()

        gene_sums = df_stress_raw.sum(axis=1)
        df_filt = df_stress_raw.loc[gene_sums >= 10].copy()
        df_cpm = df_filt.div(df_filt.sum(axis=0), axis=1) * 1e6
        df_log = np.log2(df_cpm + 1.0)

        de_rows = []
        for gene in df_log.index:
            a_vals = df_log.loc[gene, treat].values.astype(float)
            b_vals = df_log.loc[gene, ctrl].values.astype(float)
            a_vals = a_vals[np.isfinite(a_vals)]
            b_vals = b_vals[np.isfinite(b_vals)]
            if len(a_vals) < 2 or len(b_vals) < 2: continue
            log2fc = float(np.mean(a_vals) - np.mean(b_vals))
            try:
                t_stat, pval = stats.ttest_ind(a_vals, b_vals, equal_var=False)
            except Exception:
                t_stat, pval = 0.0, 1.0
            de_rows.append({"gene": gene, "log2FC": log2fc, "t_stat": float(t_stat), "pval": float(pval)})
        df_de = pd.DataFrame(de_rows)
        df_de["padj"] = bh_fdr(df_de["pval"].values)

        # Three alternative signature definitions
        sigs = {}
        sigs["curated"] = {"up": GR_CORE, "down": []}
        # by |t-stat|, top 30 up + 30 down
        df_de_sorted = df_de.copy()
        df_de_sorted["abs_t"] = df_de_sorted["t_stat"].abs()
        sigs["topT"] = {
            "up":   df_de_sorted[df_de_sorted["t_stat"] > 0].sort_values("abs_t", ascending=False).head(30)["gene"].tolist(),
            "down": df_de_sorted[df_de_sorted["t_stat"] < 0].sort_values("abs_t", ascending=False).head(30)["gene"].tolist(),
        }
        # by |log2FC| no padj
        sigs["topFC"] = {
            "up":   df_de.sort_values("log2FC", ascending=False).head(50)["gene"].tolist(),
            "down": df_de.sort_values("log2FC", ascending=True).head(50)["gene"].tolist(),
        }

        def score_samples(df_log, up, down):
            up_p = [g for g in up if g in df_log.index]
            dn_p = [g for g in down if g in df_log.index]
            if len(up_p) < 3: return pd.Series(index=df_log.columns, data=np.nan)
            s_up = df_log.loc[up_p].mean(axis=0)
            s_dn = df_log.loc[dn_p].mean(axis=0) if len(dn_p) >= 3 else 0.0
            return s_up - s_dn

        def auc_score(y, s):
            y = np.asarray(y).astype(int); s = np.asarray(s).astype(float)
            m = np.isfinite(s)
            y = y[m]; s = s[m]
            if len(np.unique(y)) < 2 or y.size < 6: return np.nan
            n1 = (y == 1).sum(); n0 = (y == 0).sum()
            ranks = stats.rankdata(s)
            return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / max(n1 * n0, 1))

        a14_rows = []
        for name, sig in sigs.items():
            s = score_samples(df_log, sig["up"], sig["down"])
            y_aug = np.array([0]*len(ctrl) + [1]*len(treat))
            s_aug = np.array(list(s[ctrl].values) + list(s[treat].values))
            auc = auc_score(y_aug, s_aug)
            d_a, d_b = s[treat].dropna().values, s[ctrl].dropna().values
            if len(d_a) >= 2 and len(d_b) >= 2:
                _, p_mw = stats.mannwhitneyu(d_a, d_b, alternative="two-sided")
                cohens = float((np.mean(d_a) - np.mean(d_b)) /
                                (np.sqrt(((len(d_a)-1)*np.var(d_a, ddof=1) + (len(d_b)-1)*np.var(d_b, ddof=1)) /
                                          max(len(d_a)+len(d_b)-2, 1)) + 1e-12))
            else:
                p_mw, cohens = np.nan, np.nan
            a14_rows.append({
                "signature": name,
                "n_up": len(sig["up"]), "n_down": len(sig["down"]),
                "n_up_present": len([g for g in sig["up"] if g in df_log.index]),
                "n_down_present": len([g for g in sig["down"] if g in df_log.index]),
                "AUC_ctrl_vs_treated": auc,
                "cohens_d": cohens,
                "p_MW": p_mw,
            })
        df_a14 = pd.DataFrame(a14_rows)
        save_xlsx({"signature_comparison": df_a14}, "A14_GR_Signature_Alternatives")
        log(f"  AUCs: " + ", ".join([f"{r['signature']}={r['AUC_ctrl_vs_treated']:.3f}" for _, r in df_a14.iterrows()]))
except Exception as e:
    log(f"  A14 FAILED: {e}\n{traceback.format_exc()}")

# A15. MS drop sensitivity in meta-analysis (own #18)
# MS GSE180759 lacks donor metadata in NB4, contributes huge N (cell-level)
# to the meta-weights. Re-run random-effects meta excluding MS.
log("=" * 72)
log("A15: MS drop sensitivity in random-effects meta", tag="A15")
try:
    # Build donor-aggregated GR-MES correlation table for each cohort,
    # then run meta with and without MS.
    per_cohort_rows = []
    for ds, a in cohorts.items():
        gr_col = "GR_composite" if "GR_composite" in a.obs.columns else None
        if gr_col is None: continue
        donor_col = detect_donor_col(a.obs)
        score_cols = [gr_col] + [f"{m}_score" for m in mes_cols]
        score_cols = [c for c in score_cols if c in a.obs.columns]
        if donor_col is not None and donor_col in a.obs.columns:
            ddf = donor_aggregate(a.obs, donor_col, score_cols)
            level = "donor"
            N = len(ddf)
        else:
            ddf = a.obs[score_cols].copy()
            level = "cell"
            N = len(ddf)
        for mes in mes_cols:
            msc = f"{mes}_score"
            if msc not in ddf.columns: continue
            r = safe_corr_with_ci(ddf[gr_col], ddf[msc], method="spearman",
                                   n_boot=300, min_n=5)
            per_cohort_rows.append({
                "dataset": ds, "level": level, "N": N,
                "MES": mes, "r": r["r"], "p": r["p"],
                "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"],
            })
    df_pc = pd.DataFrame(per_cohort_rows)
    save_xlsx({"per_cohort_GR_MES": df_pc}, "A15_PerCohort_GR_MES_donor")

    # Meta: all vs no-MS
    a15_rows = []
    for mes in mes_cols:
        sub = df_pc[df_pc["MES"] == mes]
        meta_all = random_effects_meta(sub["r"].tolist(), sub["N"].tolist())
        sub_noms = sub[~sub["dataset"].astype(str).str.contains("MS", case=False, regex=False)]
        meta_no = random_effects_meta(sub_noms["r"].tolist(), sub_noms["N"].tolist())
        a15_rows.append({
            "MES": mes,
            "k_with_MS": meta_all["k"], "pooled_r_with_MS": meta_all["pooled_r"],
            "ci_lo_with": meta_all["ci_lo"], "ci_hi_with": meta_all["ci_hi"],
            "I2_with_MS": meta_all["I2"], "tau2_with_MS": meta_all["tau2"],
            "k_no_MS": meta_no["k"], "pooled_r_no_MS": meta_no["pooled_r"],
            "ci_lo_no": meta_no["ci_lo"], "ci_hi_no": meta_no["ci_hi"],
            "I2_no_MS": meta_no["I2"], "tau2_no_MS": meta_no["tau2"],
            "delta_pooled_r": (meta_no["pooled_r"] - meta_all["pooled_r"])
                if np.isfinite(meta_no["pooled_r"]) and np.isfinite(meta_all["pooled_r"]) else np.nan,
            "delta_I2": (meta_no["I2"] - meta_all["I2"])
                if np.isfinite(meta_no["I2"]) and np.isfinite(meta_all["I2"]) else np.nan,
        })
    df_a15 = pd.DataFrame(a15_rows)
    save_xlsx({"MS_drop_sensitivity": df_a15}, "A15_MS_drop_meta")
    log(f"  median |Δ pooled r| with vs without MS: {df_a15['delta_pooled_r'].abs().median():.4f}")
    log(f"  median Δ I² (no_MS - with_MS): {df_a15['delta_I2'].median():.2f}")
except Exception as e:
    log(f"  A15 FAILED: {e}\n{traceback.format_exc()}")

# Master summary
log("=" * 72)
log("MASTER: assembling summary table", tag="master")
try:
    summary_rows = [
        {"analysis": "A1", "deliverable": "Supp_Rev_A1_MES_TolScore_Overlap.xlsx",
         "addresses": "own #16 - circularity diagnostic, gene overlap MES top-50 vs tolerance-score genes"},
        {"analysis": "A2", "deliverable": "Supp_Rev_A2_LGALS3_Sensitivity.xlsx",
         "addresses": "own #15 - tolerance score without LGALS3, MES correlations recomputed"},
        {"analysis": "A3", "deliverable": "Supp_Rev_A3_TopN_Sensitivity.xlsx",
         "addresses": "R3.14 - MES top-25/50/100 cutoff sensitivity"},
        {"analysis": "A4", "deliverable": "Supp_Rev_A4_AxisPriors_Expanded.xlsx",
         "addresses": "R3.13 + own #1 - expanded axis priors hypergeometric retest"},
        {"analysis": "A5", "deliverable": "Supp_Rev_A5_NMF_FactorCrossCorr.{xlsx,png}",
         "addresses": "R3.12 - inter-module correlation matrix"},
        {"analysis": "A6", "deliverable": "Supp_Rev_A6_GeneCoverage_PerCohort.xlsx",
         "addresses": "R3.13 - gene-overlap report per validation cohort"},
        {"analysis": "A7", "deliverable": "Supp_Rev_A7_cNMF_Parameters.xlsx",
         "addresses": "R3.11 - cNMF parameters dump for Methods"},
        {"analysis": "A8", "deliverable": "Supp_Rev_A8_HVG_Sensitivity.xlsx",
         "addresses": "R3.9 - HVG count sweep 1500/2500/3500"},
        {"analysis": "A9", "deliverable": "Supp_Rev_A9_NoPriors.xlsx",
         "addresses": "R3.8 - NMF retrained on HVG only, no curated priors"},
        {"analysis": "A10", "deliverable": "Supp_Rev_A10_Microglia_Purity.xlsx",
         "addresses": "R3.7 - microglia purity per cohort, stratified MES-tolerance by purity quartile"},
        {"analysis": "A11", "deliverable": "Supp_Rev_A11_Mito_Threshold_Sensitivity.xlsx",
         "addresses": "R3.16 - QC mito threshold sensitivity 20/25/30/35%"},
        {"analysis": "A12", "deliverable": "Supp_Rev_A12_Visium_MicrogliaWeighted.{xlsx,png}",
         "addresses": "R1.2 + R3.7 - Visium microglia-weighted MES-tolerance + top-30% subset"},
        {"analysis": "A13", "deliverable": "Supp_Rev_A13_PMI_Sex_Sensitivity.xlsx",
         "addresses": "R3.18 - PMI tertile and sex stratification of GR-MES (SEA-AD)"},
        {"analysis": "A14", "deliverable": "Supp_Rev_A14_GR_Signature_Alternatives.xlsx",
         "addresses": "own #17 - three alternative GR signature derivations + AUCs"},
        {"analysis": "A15", "deliverable": "Supp_Rev_A15_MS_drop_meta.xlsx",
         "addresses": "own #18 - meta-analysis with/without MS GSE180759"},
    ]
    df_summary = pd.DataFrame(summary_rows)
    save_xlsx({"Index": df_summary}, "MASTER_Index")
    save_xlsx({"run_log": pd.DataFrame(master_log)}, "MASTER_RunLog")
    log("=" * 72)
    log("NB7 COMPLETE")
    log(f"  Figures: {FIG_DIR}")
    log(f"  Tables:  {TAB_DIR}")
except Exception as e:
    log(f"  master summary FAILED: {e}")
